In [ ]:
# Default Packges
import pandas as pd
import json, csv

# Data Prep

# Model

from transformers import (
    LongformerTokenizer, 
    LongformerForSequenceClassification,
    TrainingArguments, 
    Trainer
)

import torch
from transformers import pipeline
from datasets import Dataset


In [3]:
model_name = "allenai/longformer-base-4096"
tokenizer = LongformerTokenizer.from_pretrained(model_name)
model = LongformerForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10,          # set to your number of theme categories
    problem_type="multi_label_classification"
)

c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Monado\.cache\huggingface\hub\models--allenai--longformer-base-4096. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of LongformerForSequenceClassification were not initialized from the m

In [ ]:
def encode_lyrics(lyrics: str):
    return tokenizer(
        lyrics,
        max_length=4096,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )


# Global attention on [CLS] token (important for Longformer classification)
def forward_pass(lyrics: str):
    inputs = encode_lyrics(lyrics)
    # Longformer needs global_attention_mask — put global attention on [CLS]
    global_attention_mask = torch.zeros_like(inputs["input_ids"])
    global_attention_mask[:, 0] = 1   # [CLS] token gets global attention
    inputs["global_attention_mask"] = global_attention_mask

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.sigmoid(logits)   # multi-label: sigmoid, not softmax
    return probs

### Fine Tuning

In [ ]:

# Assume your data has columns: "lyrics", "themes" (list of labels)
df = pd.read_csv("your_lyrics_dataset.csv")
dataset = Dataset.from_pandas(df)

training_args = TrainingArguments(
    output_dir="./lyric-theme-model",
    num_train_epochs=5,
    per_device_train_batch_size=2,   # Longformer is memory-heavy; keep small
    gradient_accumulation_steps=8,   # effective batch size = 16
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                       # use if your GPU supports it
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)
trainer.train()

In [ ]:
# facebook/bart-large-mnli is the best zero-shot classifier available
# It uses NLI (natural language inference) to score each label against your text
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # GPU 0
)

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]

THRESHOLD = 0.20  # labels scoring above this are assigned — tune after inspection

def label_lyrics(lyrics: str) -> list[str]:
    # Truncate to 1024 tokens — bart-large-mnli's limit
    # For very long lyrics, we chunk and average scores
    words = lyrics.split()
    chunk_size = 400  # safe word count per chunk
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    # Accumulate scores across chunks
    score_map = {theme: 0.0 for theme in THEMES}
    for chunk in chunks:
        result = zero_shot(chunk, THEMES, multi_label=True)
        for label, score in zip(result["labels"], result["scores"]):
            score_map[label] += score

    # Average across chunks
    n = len(chunks)
    avg_scores = {t: score_map[t] / n for t in THEMES}

    # Assign labels above threshold
    assigned = [t for t, s in avg_scores.items() if s >= THRESHOLD]

    # Always assign at least the top label to avoid empty rows
    if not assigned:
        assigned = [max(avg_scores, key=avg_scores.get)]

    return assigned, avg_scores  # return scores too so you can review them


# Run over your dataset


df = pd.read_csv(r'C:\Users\Monado\Documents\Data_git\Projects\Lyrics_NLP\english_song_dataset_1.csv')  # must have a "lyrics" column

assigned_labels = []
all_scores = []

for i, row in df.iterrows():
    labels, scores = label_lyrics(row["lyrics"])
    assigned_labels.append(labels)
    all_scores.append(scores)
    if i % 50 == 0:
        print(f"Processed {i}/{len(df)} — last labels: {labels}")

df["themes"] = assigned_labels
df["theme_scores"] = all_scores
df.to_csv("lyrics_labeled.csv", index=False)
print("Done. Labeled dataset saved.")

In [ ]:
import torch
import ast
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    LongformerTokenizer,
    LongformerForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME  = "allenai/longformer-base-4096"
MAX_LENGTH  = 2048   # 16GB VRAM handles this comfortably with batch size 2
BATCH_SIZE  = 2
GRAD_ACCUM  = 8      # effective batch size = 16
EPOCHS      = 6
LR          = 2e-5

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]
NUM_LABELS = len(THEMES)
THEME_TO_IDX = {t: i for i, t in enumerate(THEMES)}

# ── Dataset class ────────────────────────────────────────────────────────────
class LyricsDataset(Dataset):
    def __init__(self, lyrics_list, labels_list, tokenizer):
        self.encodings = []
        self.labels = []

        for lyrics, themes in zip(lyrics_list, labels_list):
            enc = tokenizer(
                lyrics,
                max_length=MAX_LENGTH,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            # Global attention on [CLS] — critical for Longformer classification
            global_attn = torch.zeros(MAX_LENGTH, dtype=torch.long)
            global_attn[0] = 1

            self.encodings.append({
                "input_ids":            enc["input_ids"].squeeze(),
                "attention_mask":       enc["attention_mask"].squeeze(),
                "global_attention_mask": global_attn,
            })

            # Build multi-hot label vector
            label_vec = torch.zeros(NUM_LABELS)
            for theme in themes:
                if theme in THEME_TO_IDX:
                    label_vec[THEME_TO_IDX[theme]] = 1.0
            self.labels.append(label_vec)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {**self.encodings[idx], "labels": self.labels[idx]}


# ── Load & split data ────────────────────────────────────────────────────────
df = pd.read_csv("lyrics_labeled.csv")
df["themes"] = df["themes"].apply(ast.literal_eval)  # string → list

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

tokenizer = LongformerTokenizer.from_pretrained(MODEL_NAME)

train_dataset = LyricsDataset(train_df["lyrics"].tolist(), train_df["themes"].tolist(), tokenizer)
val_dataset   = LyricsDataset(val_df["lyrics"].tolist(),   val_df["themes"].tolist(),   tokenizer)


# ── Model ─────────────────────────────────────────────────────────────────────
model = LongformerForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro":  f1_score(labels, preds, average="micro",  zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro",  zero_division=0),
        "roc_auc":   roc_auc_score(labels, probs, average="macro"),
    }


# ── Training ──────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./lyric-theme-longformer",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    fp16=True,
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./lyric-theme-longformer/best")
tokenizer.save_pretrained("./lyric-theme-longformer/best")
print("Training complete.")

In [ ]:
def predict_themes(lyrics: str, threshold: float = 0.5) -> dict:
    model.eval()
    inputs = tokenizer(
        lyrics,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    global_attn = torch.zeros_like(inputs["input_ids"])
    global_attn[:, 0] = 1
    inputs["global_attention_mask"] = global_attn

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    results = {THEMES[i]: round(float(probs[i]), 3) for i in range(NUM_LABELS)}

    predicted = {t: s for t, s in results.items() if s >= threshold}
    return dict(sorted(predicted.items(), key=lambda x: -x[1]))

# Example
themes = predict_themes("I keep your photograph, I know it serves me well...")
print(themes)
# → {'heartbreak and loss': 0.91, 'nostalgia and memory': 0.84, 'loneliness and isolation': 0.72}